In [2]:
!pip install hydra-core --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.4 MB/s eta 0:00:00


In [3]:
%%writefile config.yaml
# config.yaml

# 1. 실험 시드 및 기본 설정
seed: 42
num_workers: 2

# 2. 데이터셋 경로
data:
  path: "/content/drive/MyDrive/Colab Notebooks/HAR_data/UCI_HAR"
  split_type: "subject"

# 3. 학습 하이퍼파라미터
train:
  batch_size: 128
  epochs: 20
  lr: 0.001
  device: "cuda"

# 4. 모델 관련 설정
model:
  num_classes: 6
  hidden_dim: 128

Writing config.yaml


In [4]:
%%writefile main.py
import os
import time
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

import hydra
from omegaconf import DictConfig, OmegaConf

# ------------------------------------------------------------------------------
# Utils
# ------------------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ------------------------------------------------------------------------------
# UCI-HAR Dataset
# ------------------------------------------------------------------------------
class UCIHARDataset(Dataset):
    def __init__(self, data_path, split='train'):
        self.split = split

        if split == 'train':
            y = np.loadtxt(os.path.join(data_path, 'train', 'y_train.txt'))
            signal_path = os.path.join(data_path, 'train', 'Inertial Signals')
        else:
            y = np.loadtxt(os.path.join(data_path, 'test', 'y_test.txt'))
            signal_path = os.path.join(data_path, 'test', 'Inertial Signals')

        signals = []
        signal_files = [
            'body_acc_x', 'body_acc_y', 'body_acc_z',
            'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
            'total_acc_x', 'total_acc_y', 'total_acc_z'
        ]

        for signal_file in signal_files:
            filename = os.path.join(signal_path, f'{signal_file}_{split}.txt')
            signal_data = np.loadtxt(filename)
            signals.append(signal_data)

        self.X = np.stack(signals, axis=-1).astype(np.float32)
        self.y = (y - 1).astype(np.int64)

        print(f"[{split}] X: {self.X.shape}, y: {self.y.shape}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.X[idx]), torch.LongTensor([self.y[idx]])[0]

# ------------------------------------------------------------------------------
# Normalizer
# ------------------------------------------------------------------------------
class Normalizer:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X_train):
        self.mean_ = X_train.mean(axis=(0, 1))
        self.std_  = X_train.std(axis=(0, 1))
        self.std_  = np.maximum(self.std_, self.eps)

    def transform(self, X):
        return (X - self.mean_) / self.std_

# ------------------------------------------------------------------------------
# Model
# ------------------------------------------------------------------------------
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes=6, hidden_dim=128):
        super().__init__()
        self.conv1 = nn.Conv1d(9, 32, 5, 3)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, 3, 1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lstm = nn.LSTM(64, hidden_dim, num_layers=1, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        logits = self.fc(x)
        return logits

# ------------------------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)
        logits = model(X)
        pred = logits.argmax(dim=1)

        y_true.append(y.cpu().numpy())
        y_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return float(f1_score(y_true, y_pred, average='macro'))

# ------------------------------------------------------------------------------
# Training Logic
# ------------------------------------------------------------------------------
def run_training(cfg):
    set_seed(cfg.seed)
    device = cfg.train.device if torch.cuda.is_available() else "cpu"
    print(f"Device: {device} | LR: {cfg.train.lr} | Batch: {cfg.train.batch_size}")

    train_ds = UCIHARDataset(cfg.data.path, "train")
    test_ds  = UCIHARDataset(cfg.data.path, "test")

    train_loader = DataLoader(
        train_ds, batch_size=cfg.train.batch_size, shuffle=True,
        num_workers=cfg.num_workers
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.train.batch_size, shuffle=False,
        num_workers=cfg.num_workers
    )

    model = CNN_LSTM(
        num_classes=cfg.model.num_classes,
        hidden_dim=cfg.model.hidden_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train.lr)
    criterion = nn.CrossEntropyLoss()

    for ep in range(1, cfg.train.epochs + 1):
        model.train()
        loss_sum = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * X.size(0)

        if ep % 5 == 0 or ep == cfg.train.epochs:
            train_loss = loss_sum / len(train_ds)
            f1 = evaluate(model, test_loader, device)
            print(f"[Epoch {ep:02d}] Loss: {train_loss:.4f} | F1: {f1:.4f}")

# ------------------------------------------------------------------------------
# Main with Hydra
# ------------------------------------------------------------------------------
@hydra.main(config_path=".", config_name="config", version_base=None)
def main(cfg: DictConfig):
    print(OmegaConf.to_yaml(cfg))
    run_training(cfg)

if __name__ == "__main__":
    main()

Writing main.py


In [5]:
!python main.py

seed: 42
num_workers: 2
data:
  path: /content/drive/MyDrive/Colab Notebooks/HAR_data/UCI_HAR
  split_type: subject
train:
  batch_size: 128
  epochs: 20
  lr: 0.001
  device: cuda
model:
  num_classes: 6
  hidden_dim: 128

Device: cuda | LR: 0.001 | Batch: 128
[train] X: (7352, 128, 9), y: (7352,)
[test] X: (2947, 128, 9), y: (2947,)
[Epoch 05] Loss: 0.1396 | F1: 0.9105
[Epoch 10] Loss: 0.1293 | F1: 0.9131
[Epoch 15] Loss: 0.1051 | F1: 0.9134
[Epoch 20] Loss: 0.1036 | F1: 0.9132


In [7]:
!python main.py train.lr=0.01 train.epochs=5

seed: 42
num_workers: 2
data:
  path: /content/drive/MyDrive/Colab Notebooks/HAR_data/UCI_HAR
  split_type: subject
train:
  batch_size: 128
  epochs: 5
  lr: 0.01
  device: cuda
model:
  num_classes: 6
  hidden_dim: 128

Device: cuda | LR: 0.01 | Batch: 128
[train] X: (7352, 128, 9), y: (7352,)
[test] X: (2947, 128, 9), y: (2947,)
[Epoch 05] Loss: 0.1312 | F1: 0.8940


In [9]:
!python main.py -m train.lr=0.001,0.0002

[2026-02-18 12:48:49,256][HYDRA] Launching 2 jobs locally
[2026-02-18 12:48:49,256][HYDRA] 	#0 : train.lr=0.001
seed: 42
num_workers: 2
data:
  path: /content/drive/MyDrive/Colab Notebooks/HAR_data/UCI_HAR
  split_type: subject
train:
  batch_size: 128
  epochs: 20
  lr: 0.001
  device: cuda
model:
  num_classes: 6
  hidden_dim: 128

Device: cuda | LR: 0.001 | Batch: 128
[train] X: (7352, 128, 9), y: (7352,)
[test] X: (2947, 128, 9), y: (2947,)
[Epoch 05] Loss: 0.1396 | F1: 0.9105
[Epoch 10] Loss: 0.1293 | F1: 0.9131
[Epoch 15] Loss: 0.1051 | F1: 0.9134
[Epoch 20] Loss: 0.1036 | F1: 0.9132
[2026-02-18 12:49:05,842][HYDRA] 	#1 : train.lr=0.0002
seed: 42
num_workers: 2
data:
  path: /content/drive/MyDrive/Colab Notebooks/HAR_data/UCI_HAR
  split_type: subject
train:
  batch_size: 128
  epochs: 20
  lr: 0.0002
  device: cuda
model:
  num_classes: 6
  hidden_dim: 128

Device: cuda | LR: 0.0002 | Batch: 128
[train] X: (7352, 128, 9), y: (7352,)
[test] X: (2947, 128, 9), y: (2947,)
[Epoch 05